In [2]:
from pathlib import Path

import xarray as xr
import panel as pn
from holoviews import opts

import echoshader


In [3]:
path_MVBS = Path("/media/volume/shimada_202506_volume/viz_data_cache")
MVBS_files = sorted(list(path_MVBS.glob("MVBS_*.zarr")))
MVBS_files

[PosixPath('/media/volume/shimada_202506_volume/viz_data_cache/MVBS_latest.zarr')]

In [4]:
ds_MVBS = xr.open_zarr(MVBS_files[0])
ds_MVBS["echo_range"] = ds_MVBS["depth"]
ds_MVBS = ds_MVBS.swap_dims({"depth": "echo_range"})
ds_MVBS["Sv"] = ds_MVBS["Sv"].assign_attrs(
    actual_range=(float(ds_MVBS["Sv"].min().compute()),
                  float(ds_MVBS["Sv"].max().compute()))
)

In [5]:
def plot_multi_freq():
    egram = ds_MVBS.eshader.echogram(
        channel=[
            "WBT 400141-15 ES18_ES",
            "WBT 400143-15 ES38B_ES",
            "WBT 400142-15 ES70-7C_ES",
            "WBT 400140-15 ES120-7C_ES",
            "WBT 400145-15 ES200-7C_ES",
        ],
        vmin=-70,
        vmax=-36,
        cmap = "viridis",
        opts = opts.Image(
            width=800, height=400,
            tools=["pan", "box_zoom", "wheel_zoom", "reset"],
        )
    )
    return egram

In [6]:
update_button = pn.widgets.Button(name='Refresh data')

def update_plot(event=None):   
    return(plot_multi_freq())

view_multi_freq = pn.Column(
    update_button,
    pn.panel(pn.bind(update_plot, update_button), loading_indicator=True),
)

In [7]:
test_server = pn.serve(
    {"test": view_multi_freq},
    port=1803,
    websocket_origin="*",
    admin=True,
    show=False,
)

Launching server at http://localhost:1803


In [7]:
test_server.stop()